# Step 2: Process Features
Ce notebook évalue sur chaque segment du réseaux chaque attribut. La méthode de comptabilisation de chaque attribut est prédéfinie dans le fichier excel attributs_info.

## Traitement des attributs

In [1]:
%load_ext autoreload
%autoreload 2


import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os, time
import numpy as np
import geopandas as gpd
import shapely


import os

# from method_a_buffer import extract_buffer_feature
from feature_to_network import extract_buffer_feature
from feature_query import make_feature_query

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm
import rasterio
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)


operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "walk"  # walk or bike


input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3"
save_path = f"../../Data/output/{network}/{territory}/step-2"
attribute_source_path = f"source_path_{territory}"
file_name = f"file_name_{territory}"  # column name in attributs_info excel
how_name = f"how_{territory}"
geometry_type_name = f"geometry_type_{territory}"
value_column_name = f"value_column_{territory}"
include = f"include_in_index_{territory}"

# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)


Set parameters : GG or GE, bike or walk
Loading pedestrian segments...


In [ ]:
# Classeur par réseau (mettre à jour si le fichier bike WIP est renommé)
ATTRIBUTS_INFO_FILE = {"walk": "attributs_info_walk_urbain.xlsx",
                       "bike": "attributs_info_bike_WIP.xlsx"}[network]
attributs_info = pd.read_excel(f"{input_file_path}/attributs/{ATTRIBUTS_INFO_FILE}", sheet_name="attributs_info")

# Une ligne est traitée en étape 2 si elle est dans l'indice (include_in_index_<T> = True)
# OU marquée compute_step2_<T> = True (ex. comptages intermédiaires n_traversees, n_traversees_feux).
# Excel peut lire ces colonnes en booléens ou en texte : on normalise.
def _is_true(col):
    return col.astype(str).str.strip().str.lower().eq("true")

compute_name = f"compute_step2_{territory}"
_in_index = _is_true(attributs_info[include])
_compute = _is_true(attributs_info[compute_name]) if compute_name in attributs_info.columns else False
attributs_info = attributs_info.loc[_in_index | _compute].copy()

print(f"Lignes traitées en étape 2 : {len(attributs_info)} "
      f"(dont hors indice : {int((~_is_true(attributs_info[include])).sum())})")
attributs_info

In [3]:
segmented_net

,u,v,key,walk_class,source,highway,osm_id,id,element,footway,sidewalk,name,maxspeed,service,access,foot,platform_type,tracktype,surface,smoothness,len_m,walk_role,proxy_dedicated_covered_len_m,proxy_dedicated_coverage_ratio,edge_id,adj_osm_id,adj_highway,adj_speed_kmh,adj_dist_m,speed_ctx_kmh,length,geometry,segment_id
0,0,3996,0,walk_dedicated,OSM_highway_dedicated,footway,way/4272878,4.272878e+06,way,None,None,None,None,None,None,None,None,None,concrete,None,5.920195,dedicated,NaN,NaN,0,None,None,NaN,NaN,NaN,5.920195,"LINESTRING (2498336.577 1117565.488, 2498341.8...",000000
1,1,2,0,walk_track,OSM_highway_dedicated,track,way/4297439,4.297439e+06,way,None,None,Chemin du Milieu,None,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,None,None,NaN,NaN,NaN,50.000000,"LINESTRING (2495505.257 1128846.192, 2495471.4...",000001
2,1,2,0,walk_track,OSM_highway_dedicated,track,way/4297439,4.297439e+06,way,None,None,Chemin du Milieu,None,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,None,None,NaN,NaN,NaN,50.000000,"LINESTRING (2495471.423 1128883.007, 2495437.5...",000002
3,1,2,0,walk_track,OSM_highway_dedicated,track,way/4297439,4.297439e+06,way,None,None,Chemin du Milieu,None,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,None,None,NaN,NaN,NaN,50.000000,"LINESTRING (2495437.589 1128919.821, 2495403.7...",000003
4,1,2,0,walk_track,OSM_highway_dedicated,track,way/4297439,4.297439e+06,way,None,None,Chemin du Milieu,None,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,None,None,NaN,NaN,NaN,50.000000,"LINESTRING (2495403.756 1128956.635, 2495369.9...",000004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536729,237403,86667,0,sidewalk_proxy_road_axis,OSM_highway_proxy,service,way/1568023835,1.568024e+09,way,None,None,None,None,driveway,None,None,None,None,None,None,39.893638,proxy_local,21.280947,0.533442,184275,way/1568023835,service,NaN,8.106816e-11,NaN,9.291826,"LINESTRING (2498262.993 1119689.141, 2498269.4...",536729
536730,86667,113138,0,sidewalk_proxy_road_axis,OSM_highway_proxy,service,way/1568023835,1.568024e+09,way,None,None,None,None,driveway,None,None,None,None,None,None,39.893638,proxy_local,21.280947,0.533442,184275,way/1568023835,service,NaN,2.038539e-10,NaN,30.601812,"LINESTRING (2498269.463 1119682.472, 2498287.6...",536730
536731,237404,237405,0,sidewalk_proxy_road_axis,OSM_highway_proxy,service,way/1568023836,1.568024e+09,way,None,None,None,None,driveway,None,None,None,None,None,None,40.056797,proxy_local,20.177314,0.503718,184276,way/1568023836,service,NaN,0.000000e+00,NaN,5.591375,"LINESTRING (2498259.114 1119685.626, 2498262.9...",536731
536732,237405,113139,0,sidewalk_proxy_road_axis,OSM_highway_proxy,service,way/1568023836,1.568024e+09,way,None,None,None,None,driveway,None,None,None,None,None,None,40.056797,proxy_local,20.177314,0.503718,184276,way/1568023836,service,NaN,4.113893e-11,NaN,34.465421,"LINESTRING (2498262.974 1119681.581, 2498283.6...",536732


V1 Original

In [4]:
# # Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
# segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()

# '''# --- ADD THIS BLOCK ---
# # Sample a subset of the network for faster testing
# sample_fraction = 0.7  # 70% of all segments, adjust as needed
# segmented_net = segmented_net.sample(frac=sample_fraction, random_state=42)

# print(f"Testing on a sample of {len(segmented_net)} segments out of the total network.")
# # ----------------------'''

# print('Boucle sur chaque attribut... peut prendre du temps (30 mn)')

# # Boucle sur le derniers attributs, first 3 only for testing
# for _, row in attributs_info.iterrows():
#     if row[include]:
#         attribute_name = row['attribute']
#         method = row['method']
#         how = row['how']
#         value_column = row['value_column']
#         buffer_size = row['buffer_size']
#         geometry_type = row['geometry_type']
#         feature_query_expr = make_feature_query(
#         row.get('filter_column'),
#         row.get('filter_values')
#     )

#         # Charger la couche attribut depuis gpd_attributs
#         print(f"Chargement de la couche: {attribute_name}")
#         attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
#         attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

#         # Appliquer la méthode
        
#         if method == "A": # Buffer feature extraction
#             print(f"Applying method {method} for attribute {attribute_name}...")
#             attribute_df = extract_buffer_feature(
#                 segments_gdf = segmented_net,
#                 feature_gdf = attribute_gdf,
#                 feature_name=attribute_name,
#                 geom_kind=geometry_type,
#                 buffer_radius=buffer_size,
#                 how=how,
#                 value_column=value_column,
#                 crs_meter_epsg=operation_crs,
#                 feature_query  = feature_query_expr,
#             )
#             print(f"Buffer feature extracted for {attribute_name} with method {method}")
        
#                 # Debug duplicates
#             if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
#                 print("\nDEBUG: Found duplicate segment assignments")
#                 dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
#                 print(f"Number of segments with multiple zones: {len(dupes['segment_id'].unique())}")
            
#                 # Keep only the first occurrence for each segment_id
#                 attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
#                 print("Dropped duplicates, keeping first occurrence")
#             print(f"Spatial join computed for {attribute_name} with method {method}") 
        
#         # Ajoute d'autres méthodes si besoin
        
#         # Ajouter la colonne au GeoDataFrame principal
#         segmented_net[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name].fillna(0)
#         print(f"Attribute {attribute_name} added to segmented_net.")

# # Sauvegarder
# print("Saving step 2 features to parquet...")
# segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)


V2

In [5]:
# # Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
# segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()

# '''# --- ADD THIS BLOCK ---
# # Sample a subset of the network for faster testing
# sample_fraction = 0.7  # 70% of all segments, adjust as needed
# segmented_net = segmented_net.sample(frac=sample_fraction, random_state=42)

# print(f"Testing on a sample of {len(segmented_net)} segments out of the total network.")
# # ----------------------'''

# print('Boucle sur chaque attribut... peut prendre du temps (30 mn)')

# # Boucle sur le derniers attributs, first 3 only for testing
# for idx, row in attributs_info.iterrows():
#     if row[include]:
#         attribute_name = row['attribute']
#         method = row['method']
#         how = row['how']
#         value_column = row['value_column']
#         buffer_size = row['buffer_size']
#         geometry_type = row['geometry_type']
        
#         try:
#             feature_query_expr = make_feature_query(
#                 row.get('filter_column'),
#                 row.get('filter_values')
#             )

#             # Charger la couche attribut depuis gpd_attributs
#             print(f"\n[{idx+1}/{len(attributs_info)}] Chargement de la couche: {attribute_name}")
#             attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
#             print(f"  → {len(attribute_gdf)} features chargées")
#             attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

#             # Appliquer la méthode
#             if method == "A": # Buffer feature extraction
#                 print(f"  → Applying method {method} (how={how}, buffer={buffer_size}m)...")
#                 attribute_df = extract_buffer_feature(
#                     segments_gdf = segmented_net,
#                     feature_gdf = attribute_gdf,
#                     feature_name=attribute_name,
#                     geom_kind=geometry_type,
#                     buffer_radius=buffer_size,
#                     how=how,
#                     value_column=value_column,
#                     crs_meter_epsg=operation_crs,
#                     feature_query  = feature_query_expr,
#                 )
#                 print(f"  → Buffer feature extracted")
            
#                 # Debug duplicates
#                 if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
#                     print("  → WARNING: Found duplicate segment assignments")
#                     dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
#                     print(f"     Segments with duplicates: {len(dupes['segment_id'].unique())}")
                
#                     # Keep only the first occurrence for each segment_id
#                     attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
#                     print("  → Duplicates dropped (keeping first)")
            
#             # Ajoute d'autres méthodes si besoin
            
#             # Ajouter la colonne au GeoDataFrame principal
#             col_name = f'{attribute_name}_{method}_{buffer_size}'
#             segmented_net[col_name] = attribute_df[attribute_name].fillna(0)
#             print(f"  ✓ Attribute {attribute_name} added to segmented_net as '{col_name}'")
            
#         except Exception as e:
#             print(f"  !!! ERROR processing {attribute_name}: {e}")
#             import traceback
#             traceback.print_exc()
#             # Continue avec l'attribut suivant au lieu de crasher
#             continue

# # Sauvegarder
# print("\n" + "="*60)
# print("Saving step 2 features to parquet...")
# segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)
# print(f"✓ Saved {len(segmented_net)} segments with {len(segmented_net.columns)} columns")
# print("="*60)

V3

In [ ]:
import os, time
import numpy as np
import geopandas as gpd
import shapely
from feature_query import make_feature_query  # ta fonction inline
from feature_to_network import extract_length_weighted_feature
from feature_rectangle import extract_rectangle_weighted_mean

# ========= Réglages de performance =========
BUFFER_RESOLUTION = 2           # précision suffisante
UNION_LINES_FOR_LENGTH_RATIO = True  # éviter le double comptage
ROUND_DECIMALS = 3              # arrondi à 3 décimales pour tous les floats

# ========= Réglage how = "weighted_length" (moyenne pondérée par la longueur) =========
# Part minimale de la longueur d'un tronçon couverte par des carreaux valides.
# En dessous -> valeur NaN (conservée jusqu'à l'étape 3, jamais remplacée par 0).
MIN_VALID_SHARE = 0.5

# ========= Réglage how = "rectangle_weighted_mean" (ex. charge TIM) =========
# Rectangle à extrémités plates, demi-largeur = buffer_size_<T> ; moyenne des lignes
# pondérée par leur longueur dans le rectangle (feature_rectangle.py).
RECT_MAX_ANGLE_DEG = 45        # lignes plus obliques que ça écartées (None = pas de filtre)
RECT_MIN_COVERED_SHARE = 0.0   # part couverte minimale, sinon NaN (0 = désactivé)

# Modes qui gardent les NaN jusqu'à l'étape 3 et ajoutent une colonne <col>_valid_share
NAN_KEEPING_HOWS = ("weighted_length", "rectangle_weighted_mean")

# ========= Préparer le réseau (CRS + colonnes utiles) =========
if segmented_net.crs is None or segmented_net.crs.to_epsg() != operation_crs:
    segmented_net = segmented_net.to_crs(operation_crs)

segmented_net = segmented_net[['segment_id', 'geometry', 'length']].copy()
segmented_net = segmented_net.drop_duplicates('segment_id')
segmented_net = segmented_net.set_index('segment_id', drop=False)

print('Boucle sur chaque attribut (méthode définie dans la table attributaire)…')


for _, row in attributs_info.iterrows():
    # Sélection des lignes faite à l'import (include_in_index OU compute_step2)

    attribute_name = row['attribute']
    how            = row[how_name]                     # presence / count / count_rectangle / mean / sum / area_ratio / length_ratio / raster / weighted_length / rectangle_weighted_mean / derived
    if how == "derived":
        # Calculé après la boucle, section « Attributs dérivés »
        print(f"\n→ {attribute_name} | how=derived : voir section « Attributs dérivés »")
        continue
    value_column   = row.get(value_column_name, None)
    buffer_size    = float(row[f"buffer_size_{territory}"])
    if how == "weighted_length" and np.isnan(buffer_size):
        buffer_size = 0.0                              # pas de buffer pour ce mode
    geometry_type  = str(row[geometry_type_name]).lower().strip()
    extend_m       = row.get(f"extend_m_{territory}", np.nan)   # utilisé seulement par count_rectangle
    rectangle_extend = float(extend_m) if pd.notna(extend_m) else 0.0

    print(f"\n→ {attribute_name} | how={how}, geom={geometry_type}, buffer={buffer_size}m")
    t0 = time.perf_counter()

    # 1) Charger la couche attribut + CRS
    attr_path = f"{output_step2_path}/parquet_attributs/{attribute_name}.parquet"
    attribute_gdf = gpd.read_parquet(attr_path)
    if attribute_gdf.crs != segmented_net.crs:
        attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

    # 2) Filtrer via make_feature_query (inline)
    expr = make_feature_query(row.get('filter_column'), row.get('filter_values'))
    if expr:
        try:
            attribute_gdf = attribute_gdf.query(expr)
        except Exception as e:
            print(f"  ⚠️ Filtre ignoré ({expr}) : {e}")

    # 3) Nettoyage géométries
    attribute_gdf = attribute_gdf[
        attribute_gdf.geometry.notna() & ~attribute_gdf.geometry.is_empty
    ].copy()
    if not attribute_gdf.geometry.is_valid.all():
        attribute_gdf['geometry'] = shapely.make_valid(attribute_gdf.geometry.values)

    print(f"  Taille attribut: {len(attribute_gdf):,} | chargement+filtre: {time.perf_counter()-t0:.1f}s")

    # 5) value_column peut être NaN → None
    if isinstance(value_column, float) and np.isnan(value_column):
        value_column = None

    # 6) Extraction
    t1 = time.perf_counter()
    if how == "weighted_length":
        # Carreaux raster vectorisés (2_1) : moyenne pondérée par la longueur traversée, sans buffer
        attribute_df = extract_length_weighted_feature(
            segments_gdf    = segmented_net.reset_index(drop=True),
            cells_gdf       = attribute_gdf,
            feature_name    = attribute_name,
            value_column    = value_column,
            min_valid_share = MIN_VALID_SHARE,
            crs_meter_epsg  = operation_crs,
        )
    elif how == "rectangle_weighted_mean":
        # Lignes (ex. charges TIM) dans un rectangle à extrémités plates, moyenne pondérée par la longueur
        attribute_df = extract_rectangle_weighted_mean(
            segments_gdf      = segmented_net.reset_index(drop=True),
            lines_gdf         = attribute_gdf,
            feature_name      = attribute_name,
            value_column      = value_column,
            half_width        = buffer_size,
            max_angle_deg     = RECT_MAX_ANGLE_DEG,
            min_covered_share = RECT_MIN_COVERED_SHARE,
            crs_meter_epsg    = operation_crs,
        )
    else:
        attribute_df = extract_buffer_feature(
            segments_gdf        = segmented_net.reset_index(drop=True),
            feature_gdf         = attribute_gdf,
            feature_name        = attribute_name,
            geom_kind           = geometry_type,
            how                 = how,
            buffer_radius       = buffer_size,
            value_column        = value_column,
            crs_meter_epsg      = operation_crs,
            predicate           = row.get('predicate', None),
            feature_query       = None,  # déjà filtré
            segment_length_col  = 'length',
            zero_for_missing    = True,
            raster_stats        = row.get('raster_stats', 'mean'),
            buffer_resolution   = BUFFER_RESOLUTION,
            rectangle_extend    = rectangle_extend,
        )
    print(f"  ✓ extraction ({how}): {time.perf_counter()-t1:.1f}s — {attribute_df.shape[0]} lignes")

    # 7) Sécurité unicité
    if attribute_df.duplicated('segment_id').any():
        ndup = attribute_df[attribute_df.duplicated('segment_id', keep=False)]['segment_id'].nunique()
        print(f"  ⚠️ Duplicates détectés pour {ndup} segments → on garde le premier")
        attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')

    # 8) Assignation + arrondi
    attribute_df = attribute_df.set_index('segment_id')
    colname = f"{attribute_name}_{int(buffer_size)}"  # plus de “_A_” ici

    if how in NAN_KEEPING_HOWS:
        # NaN conservés : un tronçon sans couverture suffisante reste NaN (traité à l'étape 3)
        if colname not in segmented_net.columns:
            segmented_net[colname] = np.nan
        share_col = f"{colname}_valid_share"
        if share_col not in segmented_net.columns:
            segmented_net[share_col] = np.nan
    elif colname not in segmented_net.columns:
        segmented_net[colname] = 0.0

    ix = segmented_net.index.intersection(attribute_df.index)
    vals = attribute_df.loc[ix, attribute_name].to_numpy(dtype='float64')
    segmented_net.loc[ix, colname] = np.around(vals, ROUND_DECIMALS)
    if how in NAN_KEEPING_HOWS:
        shares = attribute_df.loc[ix, f"{attribute_name}_valid_share"].to_numpy(dtype='float64')
        segmented_net.loc[ix, share_col] = np.around(shares, ROUND_DECIMALS)
        print(f"  → {int(segmented_net[colname].isna().sum()):,} tronçons NaN dans {colname} (gardés pour l'étape 3)")

    print(f"  → Colonne ajoutée: {colname} (arrondie à {ROUND_DECIMALS} décimales)")

# 9) Sauvegarde (et arrondi global final)
print("\nSaving step 2 features to parquet…")
if segmented_net.crs != target_crs:
    segmented_net = segmented_net.to_crs(target_crs)

float_cols = segmented_net.select_dtypes(include=['float', 'float32', 'float64']).columns
segmented_net[float_cols] = segmented_net[float_cols].astype('float64').round(ROUND_DECIMALS)

out_path = os.path.join(output_step2_path, "step2_features.parquet")
segmented_net.to_parquet(out_path, index=False)
print(f"✔️ Enregistré: {out_path}")

## Attributs dérivés

Lignes de `attributs_info` avec `how_<territoire> = derived`. Elles sont sautées par la boucle V3 et calculées ici, à partir de colonnes déjà présentes dans `step2_features`, par la fonction du même nom dans `derived_attributes.DERIVED_ATTRIBUTES`.

La cellule relit et réécrit `step2_features.parquet` : elle est autonome et peut être déplacée telle quelle au début de l'étape 3. À lancer **après** la boucle V3 et **avant** l'export gpkg.

Aujourd'hui : `traversees` = `1 − besoin × (1 − offre)`, à partir de `n_traversees_10`, `n_traversees_feux_10` et d'une vitesse par segment (`SPEED_SOURCE`).

In [7]:
# ============================================================================
# Attributs dérivés (how = "derived")
# ============================================================================
import importlib
import derived_attributes
importlib.reload(derived_attributes)
from derived_attributes import DERIVED_ATTRIBUTES

# --- Réglages ---------------------------------------------------------------
# Vitesse utilisée pour le besoin en traversées :
#   "edge"      : vitesse propre de l'arête du segment (vitesse_all_edges.parquet, jointure edge_id)
#                 = comportement historique de 1_1, identique au contrôle
#   "attribute" : colonne de l'attribut vitesse de step2_features (moyenne dans un buffer)
SPEED_SOURCE = "edge"
speed_edges_path = f"{input_file_path}/attributs/GG/vitesse/vitesse_all_edges.parquet"

features_path = os.path.join(output_step2_path, "step2_features.parquet")
features = gpd.read_parquet(features_path)


def colname(attr):
    """Nom de colonne step2 d'un attribut : même convention que la boucle V3."""
    r = attributs_info.loc[attributs_info["attribute"] == attr]
    if r.empty:
        raise KeyError(f"'{attr}' absent des lignes traitées en étape 2 (include_in_index / compute_step2)")
    return f"{attr}_{int(float(r.iloc[0][f'buffer_size_{territory}']))}"


# --- Vitesse par segment -----------------------------------------------------
if SPEED_SOURCE == "edge":
    _seg = pd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"),
                           columns=["segment_id", "edge_id"]).drop_duplicates("segment_id")
    _spd = pd.read_parquet(speed_edges_path, columns=["edge_id", "speed_kph"]).drop_duplicates("edge_id")
    segment_speed = _seg.merge(_spd, on="edge_id", how="left").set_index("segment_id")["speed_kph"]
elif SPEED_SOURCE == "attribute":
    segment_speed = features.set_index("segment_id")[colname("vitesse")]
else:
    raise ValueError(f"SPEED_SOURCE inconnu : {SPEED_SOURCE}")
print(f"Vitesse ({SPEED_SOURCE}) : {int(segment_speed.isna().sum()):,} segments sans valeur")

# --- Calcul ------------------------------------------------------------------
derived_rows = attributs_info.loc[attributs_info[how_name].astype(str).str.strip().eq("derived")]
for _, row in derived_rows.iterrows():
    name = row["attribute"]
    func = DERIVED_ATTRIBUTES.get(name)
    if func is None:
        print(f"⚠️ {name} : aucune fonction dans derived_attributes.DERIVED_ATTRIBUTES, ignoré")
        continue

    out_name = colname(name)
    ctx = {"colname": colname, "segment_speed": segment_speed, "out_name": out_name,
           "row": row, "territory": territory}
    new_cols = func(features, ctx)

    # Remplace les colonnes d'un run précédent ; NaN conservés (jamais remplacés par 0)
    features = features.drop(columns=[c for c in new_cols.columns if c in features.columns]).join(new_cols)
    print(f"\n→ {name} -> {list(new_cols.columns)}")
    print(f"  {int(features[out_name].isna().sum()):,} segments NaN dans {out_name}")
    print(features[out_name].describe().round(3).to_string())

# --- Sauvegarde --------------------------------------------------------------
_float_cols = features.select_dtypes(include=["float"]).columns
features[_float_cols] = features[_float_cols].astype("float64").round(ROUND_DECIMALS)
features.to_parquet(features_path, index=False)
print(f"\n✔️ Enregistré : {features_path}")

Vitesse (edge) : 0 segments sans valeur

→ traversees -> ['traversees_10', 'traversees_10_offre', 'traversees_10_besoin', 'traversees_10_vitesse_classe']
  0 segments NaN dans traversees_10
count    536734.000
mean          0.742
std           0.258
min           0.000
25%           0.700
50%           0.700
75%           1.000
max           1.000

✔️ Enregistré : ../../Data/output/walk/GG/step-2/step2_features.parquet


### Contrôle traversées

Compare `step2_features` (boucle V3 + attribut dérivé) au contrôle de 1_1 (`traversees_control/traversees_control_segments.parquet`). Avec `SPEED_SOURCE = "edge"`, les différences attendues sont nulles, sauf pour une traversée posée exactement sur le bord d'un rectangle (`intersects` en 2_2, `within` en 1_1).

In [8]:
# Contrôle : step2 (count_rectangle + derived) vs 1_1 (référence)
ctrl_path = f"../../Data/output/{network}/{territory}/traversees_control/traversees_control_segments.parquet"

if not os.path.exists(ctrl_path):
    print(f"Pas de contrôle trouvé ({ctrl_path}) : lancer la section « Traversées : contrôle » de 1_1")
else:
    ctrl = pd.read_parquet(ctrl_path)
    pairs = [(colname("n_traversees"), "n_traversees"),
             (colname("n_traversees_feux"), "n_traversees_feux"),
             (colname("traversees"), "traversees")]
    cmp = features[["segment_id"] + [a for a, _ in pairs]].merge(
        ctrl[["segment_id"] + [b for _, b in pairs]], on="segment_id", how="outer", indicator=True)
    print(cmp["_merge"].value_counts().to_string())

    both = cmp[cmp["_merge"] == "both"]
    for a, b in pairs:
        x, y = both[a], both[b]
        nan_mismatch = int((x.isna() != y.isna()).sum())
        diff = int(((x - y).abs() > 1e-3).sum())
        print(f"{a:>22} vs {b:<18} : {diff:,} différences, {nan_mismatch:,} NaN non concordants")
    if SPEED_SOURCE != "edge":
        print("NB : SPEED_SOURCE != 'edge' -> des écarts sur le score sont attendus")

_merge
both          536734
left_only          0
right_only         0
       n_traversees_10 vs n_traversees       : 0 différences, 0 NaN non concordants
  n_traversees_feux_10 vs n_traversees_feux  : 0 différences, 0 NaN non concordants
         traversees_10 vs traversees         : 0 différences, 0 NaN non concordants


In [9]:

segmented_net.head(20)


,segment_id,geometry,length,lac_cours_deau_100,fontaine_100,espaces_ouverts_1,rez_actif_50,tp_150,amenite_75,bruit_100,temperature_100,conflit_md_1,environnement_ouvert_naturel_30,canopee_30,banc_100,toilette_100,air_3000,connectivite_10,n_traversees_10,n_traversees_feux_10,chemin_1,pente_10,accident_150,charge_10,charge_10_valid_share,zone_apaisee_10,zone_pietonne_1,vitesse_10,eclairage_10
segment_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
000000,000000,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",5.920,1.0,0.0,0.0,0.0,0.0,0.0,0.0,32.948,0.0,0.000,0.311,0.0,0.0,38.368,0.570,0.0,0.0,0.0,28.168,0.0,10.372,1.0,0.0,0.0,0.000,1.0
000001,000001,"LINESTRING (6.08229 46.30288, 6.08185 46.3032)",50.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,46.096,32.0,0.569,0.929,0.0,0.0,31.340,0.442,0.0,0.0,1.0,2.873,0.0,177.340,1.0,1.0,0.0,46.667,0.0
000002,000002,"LINESTRING (6.08185 46.3032, 6.0814 46.30353)",50.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,46.096,0.0,0.474,0.960,0.0,0.0,31.340,0.314,0.0,0.0,1.0,4.683,0.0,14.308,1.0,1.0,0.0,10.000,0.0
000003,000003,"LINESTRING (6.0814 46.30353, 6.08095 46.30385)",50.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,46.130,4.0,0.458,0.958,0.0,0.0,31.340,0.300,0.0,0.0,1.0,5.923,0.0,7.853,1.0,1.0,0.0,10.000,0.0
000004,000004,"LINESTRING (6.08095 46.30385, 6.08051 46.30418)",50.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45.344,0.0,0.534,0.957,0.0,0.0,31.340,0.328,0.0,0.0,1.0,5.756,0.0,2.196,1.0,1.0,0.0,10.000,0.0
000005,000005,"LINESTRING (6.08051 46.30418, 6.08006 46.30451)",50.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.682,0.0,0.781,0.950,0.0,0.0,31.340,0.370,0.0,0.0,1.0,5.205,0.0,0.190,1.0,1.0,0.0,10.000,0.0
000006,000006,"LINESTRING (6.08006 46.30451, 6.07982 46.30468)",27.158,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.646,48.0,0.796,0.805,0.0,0.0,31.340,0.711,0.0,0.0,1.0,9.117,0.0,0.060,1.0,1.0,0.0,10.000,0.0
000007,000007,"LINESTRING (6.07982 46.30468, 6.0793 46.30496)",50.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.359,52.0,0.519,0.643,0.0,0.0,31.340,0.697,0.0,0.0,1.0,9.690,0.0,0.033,1.0,1.0,0.0,10.000,0.0
000008,000008,"LINESTRING (6.0793 46.30496, 6.07899 46.30512)",30.469,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41.829,24.0,0.545,0.722,0.0,0.0,31.340,0.541,0.0,0.0,1.0,13.788,0.0,0.054,1.0,1.0,0.0,10.000,0.0


In [10]:
#import step2_features and convert to gpkg for QGIS use
segmented_net = gpd.read_parquet(os.path.join(output_step2_path, "step2_features.parquet"))
segmented_net.to_file(os.path.join(output_step2_path, "step2_features.gpkg"), driver="GPKG")
